# Representations and Costs

```{contents}
:local:
:depth: 2
```

Once an ADT contract is clear, the next question is representation: how should the data be stored so the required operations are correct and efficient?


```{index} representation; data structure
```
```{index} representation independence
```

## One Contract, Multiple Representations


The same stack ADT can be represented in more than one way:

| Representation | Main idea |
| --- | --- |
| Array-backed stack | Store items in contiguous slots and track the top index. |
| List-backed stack | Delegate storage growth to `List<T>`. |
| Linked-node stack | Store each item in a node that references the next node. |

If all three preserve the same contract, client code can use them through the same interface. This is **representation independence**.


```{index} array-backed stack
```

## Array-Backed Stack


An array-backed stack stores items from left to right. The next available index is also the current count.

```text
items: [A] [B] [C] [ ] [ ]
count: 3
top:   C
```

Pushing writes into `items[count]`, then increments `count`. Popping decrements `count`, reads the value, and clears the slot.


In [ ]:
var stack = new ArrayStack<int>();
stack.Push(10);
stack.Push(20);
Console.WriteLine(stack.Pop());
Console.WriteLine(stack.Count);

class ArrayStack<T>
{
    private T[] items = new T[4];
    public int Count { get; private set; }

    public void Push(T item)
    {
        if (Count == items.Length) Array.Resize(ref items, items.Length * 2);
        items[Count] = item;
        Count++;
    }

    public T Pop()
    {
        if (Count == 0) throw new InvalidOperationException("Stack is empty.");
        Count--;
        T value = items[Count];
        items[Count] = default!;
        return value;
    }
}

```{index} linked nodes
```
```{index} reference; linked structure
```

## Linked Nodes and References


A linked representation stores each item in a small object called a node. Each node holds a value and a reference to the next node.

```text
top -> [C | next] -> [B | next] -> [A | null]
```

The variable `top` does not contain the whole stack. It contains a reference to the first node. Following references reveals the rest of the structure.


In [ ]:
Node<string>? top = null;
top = new Node<string>("A", top);
top = new Node<string>("B", top);
top = new Node<string>("C", top);

for (Node<string>? current = top; current is not null; current = current.Next)
{
    Console.WriteLine(current.Value);
}

class Node<T>
{
    public T Value { get; }
    public Node<T>? Next { get; }

    public Node(T value, Node<T>? next)
    {
        Value = value;
        Next = next;
    }
}

## Counting Work and Big-O

```{index} Big O; input size; auxiliary space
```


Let `n` be the number of records. Count a representative operation before comparing implementations. **Big-O** describes an upper bound on how work grows with input size, ignoring fixed factors for sufficiently large inputs. It is not an elapsed time or automatically a worst-case claim.

| Bound | Example | Effect of doubling input |
|---|---|---|
| O(1) | Reading one array slot | Bounded work independent of size. |
| O(log n) | Repeatedly halving a search interval | About one extra halving. |
| O(n) | Scanning all records | About twice the work. |
| O(n log n) | A typical comparison sort | Slightly more than twice the work. |
| O(n²) | Comparing every pair | About four times the work. |

These are useful upper bounds; a linear scan is also O(n²), but O(n) is more informative. A **tight bound**, written Θ, describes the actual growth order under the stated model. Later chapters use Θ when both upper and lower growth bounds are known.


In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 0, 1, 4, 8 })
{
    int[] expenses = Enumerable.Repeat(125, n).ToArray();
    long total = 0, additions = 0;
    foreach (int cents in expenses) { total += cents; additions++; }
    Console.WriteLine($"n={n}, total={total}, additions={additions}");
}

This example counts exactly `n` additions. Array access and fixed-width arithmetic are treated as constant-cost operations. A comparison of long strings can depend on string length.

Specify the case: a search may stop at the first record or inspect all `n`. **Amortized cost** averages over a sequence of operations: geometric array growth permits O(1) amortized append even though a resizing append takes O(n). This does not make every append constant-time. **Auxiliary space** counts extra storage beyond the input; a running total takes O(1), while an additional copy takes O(n). Chapter 20 develops formal bounds and richer cost models.

Before running, predict the addition counts for sizes zero, one, four, and eight. Then explain why reading only `expenses[0]` has a different growth bound and needs a nonempty-input precondition.


```{index} operation cost
```
```{index} Big O; collection operations
```

## Operation Costs


A data structure is not just correct or incorrect. It also has costs.

| Operation | Array/List-backed stack | Linked-node stack |
| --- | --- | --- |
| `Push` | Usually O(1); occasional resize costs O(n) | O(1) |
| `Pop` | O(1) | O(1) |
| `Peek` | O(1) | O(1) |
| Random access by index | O(1), if exposed | O(n) |
| Extra memory per item | low | one extra reference per node |

The best representation depends on the workload. If a program mostly pushes and pops at one end, either representation can work. If it needs fast index access, linked nodes are the wrong fit.


## Practice


1. Draw the linked-node diagram after pushing `"red"`, `"blue"`, and `"green"` onto a stack.
2. Explain why resizing an array-backed stack is occasional O(n), even though most pushes are O(1).
3. Choose a representation for an undo history and justify it using operation costs.


## Summary


A representation is the hidden storage strategy behind an ADT contract. Arrays, lists, and linked nodes make different tradeoffs in speed, memory use, and flexibility. Data-structure design is the art of matching those tradeoffs to the operations a program performs most often.


```{rubric} Footnotes
```
[^1]: Big-O compares growth under an explicit cost model. Chapter 20 extends these foundations with formal bounds, correctness arguments, and recurrences.
